In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")

# ============================================================
# LOAD LAB DATASET
# ============================================================

lab = pd.read_csv(
    DATA_DIR /"lab.csv",
    low_memory=False
)

print("File loaded successfully.")
print("Shape:", lab.shape)

print("\nColumns:")
print(lab.columns.tolist())

lab.info()

display(lab.head())


# ============================================================
# FIRST 24 HOURS
# ============================================================

lab_24h = lab[
    (lab["labresultoffset"] >= 0) &
    (lab["labresultoffset"] <= 1440)
].copy()

print(
    "\nLab observations first 24h:",
    len(lab_24h)
)

print(
    "ICU stays with labs first 24h:",
    lab_24h["patientunitstayid"].nunique()
)


# ============================================================
# LAB TESTS AVAILABLE
# ============================================================

print("\nNumber of different lab tests:")
print(
    lab_24h["labname"].nunique()
)

print("\nMost frequent lab tests:")

display(
    lab_24h["labname"]
    .value_counts()
    .head(50)
    .to_frame("number_of_results")
)


# ============================================================
# COVERAGE BY LAB TEST
# ============================================================

total_stays = (
    lab_24h["patientunitstayid"]
    .nunique()
)

lab_coverage = (
    lab_24h[
        lab_24h["labresult"].notna()
    ]
    .groupby("labname")["patientunitstayid"]
    .nunique()
    .reset_index(
        name="ICU_stays_with_data"
    )
)

lab_coverage["coverage_pct"] = (
    lab_coverage["ICU_stays_with_data"]
    / total_stays
    * 100
).round(2)

lab_coverage = (
    lab_coverage
    .sort_values(
        "coverage_pct",
        ascending=False
    )
)

print("\nCoverage by lab:")

display(lab_coverage)


# ============================================================
# MISSING VALUES
# ============================================================

missing_lab = pd.DataFrame({
    "missing": lab_24h.isna().sum(),

    "missing_pct": (
        lab_24h.isna().mean()
        * 100
    ).round(2)
})

print("\nMissing values:")

display(
    missing_lab.sort_values(
        "missing_pct"
    )
)


# ============================================================
# DUPLICATES
# ============================================================

print(
    "\nDuplicated rows:",
    lab_24h.duplicated().sum()
)


# ============================================================
# STATISTICS BY LAB TEST
# ============================================================

lab_statistics = (
    lab_24h
    .groupby("labname")["labresult"]
    .agg([
        "count",
        "mean",
        "std",
        "min",
        "median",
        "max"
    ])
    .sort_values(
        "count",
        ascending=False
    )
)

print("\nStatistics by lab:")

display(lab_statistics)


# ============================================================
# MEASUREMENTS PER ICU STAY
# ============================================================

measurements_per_stay = (
    lab_24h
    .groupby("patientunitstayid")
    .size()
)

print(
    "\nLab measurements per ICU stay:"
)

print(
    measurements_per_stay.describe()
)


# ============================================================
# MEASUREMENTS PER LAB / ICU STAY
# ============================================================

measurements_per_lab_patient = (
    lab_24h
    .dropna(
        subset=["labresult"]
    )
    .groupby([
        "patientunitstayid",
        "labname"
    ])
    .size()
    .reset_index(
        name="n_measurements"
    )
)

print(
    "\nMeasurements per lab / ICU stay:"
)

display(
    measurements_per_lab_patient[
        "n_measurements"
    ].describe()
)

File loaded successfully.
Shape: (434660, 10)

Columns:
['labid', 'patientunitstayid', 'labresultoffset', 'labtypeid', 'labname', 'labresult', 'labresulttext', 'labmeasurenamesystem', 'labmeasurenameinterface', 'labresultrevisedoffset']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 434660 entries, 0 to 434659
Data columns (total 10 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   labid                    434660 non-null  int64  
 1   patientunitstayid        434660 non-null  int64  
 2   labresultoffset          434660 non-null  int64  
 3   labtypeid                434660 non-null  int64  
 4   labname                  434660 non-null  object 
 5   labresult                431939 non-null  float64
 6   labresulttext            434639 non-null  object 
 7   labmeasurenamesystem     415754 non-null  object 
 8   labmeasurenameinterface  415763 non-null  object 
 9   labresultrevisedoffset   434660 non-null  

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset
0,437880563,1754323,-647,3,Hct,38.30,38.3,%,%,-631
1,437880572,1754323,-647,3,platelets x 1000,181.00,181,K/mcL,k/mm cu,-631
2,437880560,1754323,-647,3,RBC,4.86,4.86,M/mcL,m/mm cu,-631
3,437880570,1754323,-647,3,-monos,8.70,8.7,%,%,-631
4,437880571,1754323,-647,3,MCHC,30.40,30.4,g/dL,%,-631



Lab observations first 24h: 98870
ICU stays with labs first 24h: 2347

Number of different lab tests:
132

Most frequent lab tests:


,number_of_results
labname,
bedside glucose,6767
potassium,3641
sodium,3447
glucose,3308
chloride,3226
creatinine,3215
BUN,3203
Hgb,3124
calcium,3098



Coverage by lab:


,labname,ICU_stays_with_data,coverage_pct
118,sodium,2092,89.14
90,creatinine,2092,89.14
11,BUN,2089,89.01
96,glucose,2075,88.41
109,potassium,2072,88.28
...,...,...,...
66,Tobramycin - peak,1,0.04
68,Tobramycin - trough,1,0.04
67,Tobramycin - random,1,0.04
114,protein S,1,0.04


... (2 outputs left out)


Duplicated rows: 0

Statistics by lab:


,count,mean,std,min,median,max
labname,,,,,,
bedside glucose,6759,159.613552,71.659095,19.0,142.0,600.0
potassium,3641,4.091890,0.697077,2.1,4.0,8.6
sodium,3447,138.211024,6.468824,103.0,138.0,170.0
glucose,3306,152.706292,82.790774,10.0,130.0,975.0
chloride,3223,104.571207,7.042952,70.0,105.0,139.0
...,...,...,...,...,...,...
Tobramycin - peak,1,5.600000,NaN,5.6,5.6,5.6
Tobramycin - trough,1,0.400000,NaN,0.4,0.4,0.4
prolactin,1,27.400000,NaN,27.4,27.4,27.4



Lab measurements per ICU stay:
count    2347.000000
mean       42.126118
std        30.942131
min         1.000000
25%        24.000000
50%        34.000000
75%        54.000000
max       417.000000
dtype: float64

Measurements per lab / ICU stay:


count    63566.000000
mean         1.544348
std          1.367870
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         25.000000
Name: n_measurements, dtype: float64

In [3]:
# ============================================================
# SEARCH FOR POTENTIALLY USEFUL LABS
# ============================================================

keywords = [
    "creatin",
    "bun",
    "urea",
    "sodium",
    "potassium",
    "glucose",
    "wbc",
    "hgb",
    "hemoglobin",
    "platelet",
    "bicarbonate",
    "hco3",
    "ph",
    "lactate",
    "bilirubin"
]

lab_names = (
    lab_24h["labname"]
    .dropna()
    .unique()
)

for keyword in keywords:

    matches = sorted([
        name
        for name in lab_names
        if keyword.lower()
        in name.lower()
    ])

    print(
        f"\n{keyword}:"
    )

    for match in matches:
        print(" -", match)


creatin:
 - creatinine
 - urinary creatinine

bun:
 - BUN

urea:

sodium:
 - sodium
 - urinary sodium

potassium:
 - potassium

glucose:
 - bedside glucose
 - glucose
 - glucose - CSF

wbc:
 - WBC x 1000
 - WBC's in body fluid
 - WBC's in cerebrospinal fluid
 - WBC's in urine

hgb:
 - Hgb

hemoglobin:
 - Carboxyhemoglobin
 - Methemoglobin
 - Oxyhemoglobin

platelet:
 - platelets x 1000

bicarbonate:
 - bicarbonate

hco3:
 - HCO3

ph:
 - -lymphs
 - Acetaminophen
 - Phenytoin
 - Theophylline
 - alkaline phos.
 - pH
 - phosphate

lactate:
 - lactate

bilirubin:
 - direct bilirubin
 - total bilirubin


In [4]:
# ============================================================
# SELECTED VARIABLES - LAB AGENT
# ============================================================

selected_labs = [
    # preencher depois de analisar coverage
    # e os nomes exatos no dataset

    "creatinine",
    "BUN",
    "sodium",
    "potassium",
    "glucose"
]

In [5]:
# ============================================================
# QUALITY CHECK - SELECTED LAB VARIABLES
# ============================================================

selected_lab_data = (
    lab_24h[
        lab_24h["labname"]
        .isin(selected_labs)
    ]
    .copy()
)

total_stays = (
    lab_24h["patientunitstayid"]
    .nunique()
)

quality_rows = []

for lab_name in selected_labs:

    subset = selected_lab_data[
        selected_lab_data["labname"]
        == lab_name
    ]

    stays_with_data = (
        subset.loc[
            subset["labresult"].notna(),
            "patientunitstayid"
        ]
        .nunique()
    )

    quality_rows.append({
        "lab": lab_name,

        "ICU_stays_with_data":
            stays_with_data,

        "coverage_pct":
            round(
                stays_with_data
                / total_stays
                * 100,
                2
            ),

        "number_of_results":
            subset["labresult"]
            .notna()
            .sum(),

        "missing_results":
            subset["labresult"]
            .isna()
            .sum()
    })


lab_quality = pd.DataFrame(
    quality_rows
)

display(lab_quality)

,lab,ICU_stays_with_data,coverage_pct,number_of_results,missing_results
0,creatinine,2092,89.14,3209,6
1,BUN,2089,89.01,3203,0
2,sodium,2092,89.14,3447,0
3,potassium,2072,88.28,3641,0
4,glucose,2075,88.41,3306,2


In [6]:
# ============================================================
# STATISTICS AND PERCENTILES
# ============================================================

lab_percentiles = (
    selected_lab_data
    .groupby("labname")["labresult"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

display(lab_percentiles)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
labname,,,,,,,,,,,,
BUN,3203.0,25.727103,20.568020,2.0,4.00,7.0,12.00,19.00,32.0,67.000,98.000,212.0
creatinine,3209.0,1.521313,1.580113,0.1,0.37,0.5,0.76,1.01,1.6,4.254,8.718,20.2
glucose,3306.0,152.706292,82.790774,10.0,60.05,79.0,103.00,130.00,175.0,304.750,469.950,975.0
potassium,3641.0,4.091890,0.697077,2.1,2.70,3.1,3.60,4.00,4.4,5.400,6.200,8.6
sodium,3447.0,138.211024,6.468824,103.0,119.00,127.3,135.00,138.00,141.0,147.000,158.000,170.0


In [7]:
# ============================================================
# MEASUREMENTS PER PATIENT
# ============================================================

results = []

for lab_name in selected_labs:

    counts = (
        selected_lab_data[
            selected_lab_data["labname"]
            == lab_name
        ]
        .dropna(
            subset=["labresult"]
        )
        .groupby(
            "patientunitstayid"
        )["labresult"]
        .count()
    )

    results.append({
        "lab": lab_name,

        "patients":
            len(counts),

        "mean_measurements":
            round(
                counts.mean(),
                1
            ),

        "median_measurements":
            round(
                counts.median(),
                1
            ),

        "min_measurements":
            counts.min(),

        "max_measurements":
            counts.max()
    })


measurements_selected = (
    pd.DataFrame(results)
)

display(
    measurements_selected
)

,lab,patients,mean_measurements,median_measurements,min_measurements,max_measurements
0,creatinine,2092,1.5,1.0,1,11
1,BUN,2089,1.5,1.0,1,11
2,sodium,2092,1.6,1.0,1,21
3,potassium,2072,1.8,1.0,1,21
4,glucose,2075,1.6,1.0,1,21


In [8]:
# ============================================================
# INSPECT EXTREME VALUES
# ============================================================

for lab_name in selected_labs:

    values = (
        selected_lab_data.loc[
            selected_lab_data["labname"]
            == lab_name,
            "labresult"
        ]
        .dropna()
    )

    print(
        "\n",
        "=" * 60
    )

    print(lab_name)

    print(
        "=" * 60
    )

    print(
        values.describe(
            percentiles=[
                0.01,
                0.05,
                0.50,
                0.95,
                0.99
            ]
        )
    )

    print("\nLowest values:")

    print(
        values
        .sort_values()
        .head(10)
        .tolist()
    )

    print("\nHighest values:")

    print(
        values
        .sort_values(
            ascending=False
        )
        .head(10)
        .tolist()
    )


creatinine
count    3209.000000
mean        1.521313
std         1.580113
min         0.100000
1%          0.370000
5%          0.500000
50%         1.010000
95%         4.254000
99%         8.718000
max        20.200000
Name: labresult, dtype: float64

Lowest values:
[0.1, 0.19, 0.2, 0.23, 0.24, 0.24, 0.26, 0.26, 0.28, 0.3]

Highest values:
[20.2, 15.22, 14.95, 14.2, 13.44, 13.42, 13.34, 13.2, 12.68, 12.14]

BUN
count    3203.000000
mean       25.727103
std        20.568020
min         2.000000
1%          4.000000
5%          7.000000
50%        19.000000
95%        67.000000
99%        98.000000
max       212.000000
Name: labresult, dtype: float64

Lowest values:
[2.0, 2.0, 2.0, 2.0, 2.0, 3.0, 3.0, 3.0, 3.0, 3.0]

Highest values:
[212.0, 202.0, 198.0, 198.0, 178.0, 175.0, 151.0, 145.0, 141.0, 129.0]

sodium
count    3447.000000
mean      138.211024
std         6.468824
min       103.000000
1%        119.000000
5%        127.300000
50%       138.000000
95%       147.000000
99%      

In [9]:

# ============================================================
# FINAL DATASET - LAB AGENT
# First 24h: first, mean, min, max, last
# Handle measurements with identical timestamps
# ============================================================

keys = ["patientunitstayid", "labname"]

# Keep only valid measurements
valid_labs = selected_lab_data.dropna(
    subset=["labresult", "labresultoffset"]
).copy()

# ============================================================
# MEAN, MIN AND MAX
# ============================================================

lab_stats = (
    valid_labs
    .groupby(keys)["labresult"]
    .agg(["mean", "min", "max"])
)

# ============================================================
# RESOLVE TIMESTAMP TIES
# Average measurements recorded at the same minute
# ============================================================

labs_by_time = (
    valid_labs
    .groupby(keys + ["labresultoffset"], as_index=False)
    ["labresult"]
    .mean()
    .sort_values("labresultoffset")
)

# ============================================================
# FIRST AND LAST MEASUREMENTS
# ============================================================

first_values = (
    labs_by_time
    .groupby(keys)["labresult"]
    .first()
    .rename("first")
)

last_values = (
    labs_by_time
    .groupby(keys)["labresult"]
    .last()
    .rename("last")
)

# ============================================================
# COMBINE ALL STATISTICS
# ============================================================

lab_summary = (
    pd.concat(
        [first_values, lab_stats, last_values],
        axis=1
    )
    .reset_index()
)

lab_summary = lab_summary[
    [
        "patientunitstayid",
        "labname",
        "first",
        "mean",
        "min",
        "max",
        "last"
    ]
]

display(lab_summary.head())

,patientunitstayid,labname,first,mean,min,max,last
0,141765,BUN,28.00,28.00,28.00,28.00,28.00
1,141765,creatinine,1.04,1.04,1.04,1.04,1.04
2,141765,glucose,61.00,61.00,61.00,61.00,61.00
3,141765,potassium,4.10,4.10,4.10,4.10,4.10
4,141765,sodium,139.00,139.00,139.00,139.00,139.00


In [10]:
# ============================================================
# PIVOT - ONE ROW PER ICU STAY
# ============================================================

lab_agent_data = (
    lab_summary
    .pivot(
        index="patientunitstayid",
        columns="labname",
        values=[
            "first",
            "mean",
            "min",
            "max",
            "last"
        ]
    )
)

lab_agent_data.columns = [
    f"{lab_name}_{stat}"
    for stat, lab_name
    in lab_agent_data.columns
]

lab_agent_data = (
    lab_agent_data
    .reset_index()
)

print(
    "Shape:",
    lab_agent_data.shape
)

display(
    lab_agent_data.head()
)

Shape: (2118, 26)


,patientunitstayid,BUN_first,creatinine_first,glucose_first,potassium_first,sodium_first,BUN_mean,creatinine_mean,glucose_mean,potassium_mean,...,BUN_max,creatinine_max,glucose_max,potassium_max,sodium_max,BUN_last,creatinine_last,glucose_last,potassium_last,sodium_last
0,141765,28.0,1.04,61.0,4.1,139.0,28.0,1.040,61.0,4.1,...,28.0,1.04,61.0,4.1,139.0,28.0,1.04,61.0,4.1,139.0
1,143870,13.0,0.89,123.0,4.4,139.0,13.5,1.015,131.5,4.4,...,14.0,1.14,140.0,4.4,139.0,14.0,1.14,140.0,4.4,133.0
2,144815,6.0,0.63,82.0,3.7,141.0,6.0,0.630,82.0,3.7,...,6.0,0.63,82.0,3.7,141.0,6.0,0.63,82.0,3.7,141.0
3,145427,14.0,1.05,139.0,4.5,141.0,14.0,1.050,139.0,4.5,...,14.0,1.05,139.0,4.5,141.0,14.0,1.05,139.0,4.5,141.0
4,147306,NaN,0.65,NaN,NaN,NaN,NaN,0.650,NaN,NaN,...,NaN,0.65,NaN,NaN,NaN,NaN,0.65,NaN,NaN,NaN


In [12]:
patient = pd.read_csv(
    DATA_DIR /"patient.csv",
    low_memory=False
)

In [13]:
# ============================================================
# INCLUDE ALL ICU STAYS
# ============================================================

lab_agent_data = (
    patient[
        ["patientunitstayid"]
    ]
    .drop_duplicates()
    .merge(
        lab_agent_data,
        on="patientunitstayid",
        how="left"
    )
)

print(
    "Final shape:",
    lab_agent_data.shape
)

Final shape: (2520, 26)


In [14]:
# ============================================================
# FINAL CHECK
# ============================================================

print(
    "Number of ICU stays:",
    len(lab_agent_data)
)

print(
    "Unique ICU stays:",
    lab_agent_data[
        "patientunitstayid"
    ].nunique()
)

print(
    "Duplicated patientunitstayid:",
    lab_agent_data[
        "patientunitstayid"
    ].duplicated().sum()
)


print("\nMissing values:")

display(
    lab_agent_data
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .to_frame(
        "missing_pct"
    )
)


print("\nColumns:")

print(
    lab_agent_data.columns.tolist()
)


print("\nFirst rows:")

display(
    lab_agent_data.head()
)


print("\nBasic statistics:")

display(
    lab_agent_data
    .describe()
    .T
)

Number of ICU stays: 2520
Unique ICU stays: 2520
Duplicated patientunitstayid: 0

Missing values:


,missing_pct
patientunitstayid,0.00
BUN_first,17.10
creatinine_first,16.98
glucose_first,17.66
potassium_first,17.78
sodium_first,16.98
BUN_mean,17.10
creatinine_mean,16.98
glucose_mean,17.66
potassium_mean,17.78



Columns:
['patientunitstayid', 'BUN_first', 'creatinine_first', 'glucose_first', 'potassium_first', 'sodium_first', 'BUN_mean', 'creatinine_mean', 'glucose_mean', 'potassium_mean', 'sodium_mean', 'BUN_min', 'creatinine_min', 'glucose_min', 'potassium_min', 'sodium_min', 'BUN_max', 'creatinine_max', 'glucose_max', 'potassium_max', 'sodium_max', 'BUN_last', 'creatinine_last', 'glucose_last', 'potassium_last', 'sodium_last']

First rows:


,patientunitstayid,BUN_first,creatinine_first,glucose_first,potassium_first,sodium_first,BUN_mean,creatinine_mean,glucose_mean,potassium_mean,...,BUN_max,creatinine_max,glucose_max,potassium_max,sodium_max,BUN_last,creatinine_last,glucose_last,potassium_last,sodium_last
0,141764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,141765,28.0,1.04,61.0,4.1,139.0,28.0,1.040,61.0,4.1,...,28.0,1.04,61.0,4.1,139.0,28.0,1.04,61.0,4.1,139.0
2,143870,13.0,0.89,123.0,4.4,139.0,13.5,1.015,131.5,4.4,...,14.0,1.14,140.0,4.4,139.0,14.0,1.14,140.0,4.4,133.0
3,144815,6.0,0.63,82.0,3.7,141.0,6.0,0.630,82.0,3.7,...,6.0,0.63,82.0,3.7,141.0,6.0,0.63,82.0,3.7,141.0
4,145427,14.0,1.05,139.0,4.5,141.0,14.0,1.050,139.0,4.5,...,14.0,1.05,139.0,4.5,141.0,14.0,1.05,139.0,4.5,141.0



Basic statistics:


,count,mean,std,min,25%,50%,75%,max
patientunitstayid,2520.0,1.666226e+06,1.022224e+06,141764.00,772262.7500,1.590372e+06,2697421.75,3.353113e+06
BUN_first,2089.0,2.518828e+01,1.920794e+01,2.00,13.0000,1.900000e+01,31.00,2.120000e+02
creatinine_first,2092.0,1.463978e+00,1.547690e+00,0.10,0.7500,1.000000e+00,1.50,2.020000e+01
glucose_first,2075.0,1.468263e+02,8.102784e+01,10.00,101.0000,1.240000e+02,163.00,9.750000e+02
potassium_first,2072.0,4.097104e+00,6.778084e-01,2.10,3.7000,4.000000e+00,4.40,8.600000e+00
sodium_first,2092.0,1.382130e+02,5.442680e+00,103.00,136.0000,1.390000e+02,141.00,1.700000e+02
BUN_mean,2089.0,2.483297e+01,1.860511e+01,2.00,13.0000,1.900000e+01,31.00,1.976000e+02
creatinine_mean,2092.0,1.444355e+00,1.481738e+00,0.10,0.7500,1.000000e+00,1.49,1.590000e+01
glucose_mean,2075.0,1.423945e+02,6.519080e+01,16.00,102.0000,1.250000e+02,160.50,8.960000e+02
potassium_mean,2072.0,4.081019e+00,6.121766e-01,2.30,3.7000,4.000000e+00,4.40,8.600000e+00


In [15]:
import os

# ============================================================
# SAVE FINAL DATASETS
# ============================================================

OUTPUT_DIR = os.path.join(
    os.getcwd(),
    "processed_data"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# Aggregated dataset
lab_agent_data.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "lab_agent_data.csv"
    ),
    index=False
)

# Raw temporal dataset
lab_agent_raw = (
    selected_lab_data[
        [
            "patientunitstayid",
            "labresultoffset",
            "labname",
            "labresult"
        ]
    ]
    .copy()
)

lab_agent_raw.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "lab_agent_raw.csv"
    ),
    index=False
)

print("Files saved successfully in processed_data:")
print("- lab_agent_data.csv")
print("- lab_agent_raw.csv")

Files saved successfully in processed_data:
- lab_agent_data.csv
- lab_agent_raw.csv
